# Audit Duplikat & Data Leakage: MD5 Hash + pHash Hamming Distance
Notebook ini mengecek apakah ada gambar duplikat atau hampir-duplikat yang bocor antar split (train/val/test).

**Dua teknik yang dipakai:**
1. **Exact Duplicate (MD5 Hash)** — mendeteksi file yang isinya 100% identik byte per byte.
2. **Near-Duplicate (pHash + Hamming Distance)** — mendeteksi gambar yang mirip secara visual meski beda resolusi, crop, atau kompresi.

**Dataset:** Irisan 3 Kelas (NV, MEL, BKL) — 22.051 citra bersih, split 80:10:10 bebas kebocoran lesi.

## Langkah 1: Install & Import Library

In [ ]:
!pip install -q imagehash

import os
import hashlib
import numpy as np
import pandas as pd
import imagehash
from PIL import Image
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display

print('Library siap.')

## Langkah 2: Muat Data CSV dan Gabungkan Split
Kita perlu tahu gambar mana masuk train, val, atau test supaya bisa cek apakah ada duplikat yang nyebrang antar split.

In [ ]:
# ============================================================
# KONFIGURASI — sesuaikan path ini dengan lokasi dataset kamu
# ============================================================
DATASET_TYPE = "irisan"  # ganti "biner" kalau mau audit dataset biner

if DATASET_TYPE == "irisan":
    CSV_NAMES = {
        "train": "dataset_irisan_3kelas_train.csv",
        "val":   "dataset_irisan_3kelas_val.csv",
        "test":  "dataset_irisan_3kelas_test.csv",
    }
else:
    CSV_NAMES = {
        "train": "dataset_binary_train.csv",
        "val":   "dataset_binary_val.csv",
        "test":  "dataset_binary_test.csv",
    }

SEARCH_DIRS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("Dataset"),
    Path("."),
]

def cari_file(nama_file):
    for d in SEARCH_DIRS:
        for root, dirs, files in os.walk(d):
            if nama_file in files:
                return Path(root) / nama_file
    return None

dfs = []
for split_name, csv_name in CSV_NAMES.items():
    path = cari_file(csv_name)
    if path is None:
        raise FileNotFoundError(f"File {csv_name} tidak ditemukan!")
    df_tmp = pd.read_csv(path)
    df_tmp["split"] = split_name
    dfs.append(df_tmp)
    print(f"  {split_name:5s} : {len(df_tmp):,} baris — {path}")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal gabungan: {len(df_all):,} citra")
print(df_all["split"].value_counts().to_string())

## Langkah 3: Cari Path Gambar yang Valid
Karena path di CSV bisa beda antara lokal Windows vs Kaggle, kita cari ulang lokasi file gambar yang benar.

In [ ]:
IMG_SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("Dataset"),
]

index_cache = {}
for root_dir in IMG_SEARCH_ROOTS:
    if not root_dir.exists():
        continue
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                index_cache[fn.split('.')[0]] = os.path.join(dirpath, fn)

print(f"Total file gambar terindeks: {len(index_cache):,}")

df_all["image_path"] = df_all["image_id"].map(index_cache)
missing = df_all["image_path"].isna().sum()

if missing > 0:
    print(f"PERINGATAN: {missing} gambar tidak ditemukan, akan diskip.")
    df_all = df_all.dropna(subset=["image_path"]).reset_index(drop=True)

print(f"Gambar siap diaudit: {len(df_all):,}")

## Langkah 4: Hitung MD5 Hash (Exact Duplicate Detection)
MD5 hash membaca seluruh isi file byte-per-byte. Kalau dua file punya hash yang sama, berarti isinya 100% identik.

In [ ]:
def hitung_md5(filepath):
    h = hashlib.md5()
    try:
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(65536), b''):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None

print(f"Menghitung MD5 hash untuk {len(df_all):,} gambar...")

md5_results = {}
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(hitung_md5, row.image_path): row.image_id
               for row in df_all.itertuples()}
    done = 0
    for fut in as_completed(futures):
        img_id = futures[fut]
        md5_results[img_id] = fut.result()
        done += 1
        if done % 5000 == 0:
            print(f"  ... {done:,}/{len(df_all):,}")

df_all["md5"] = df_all["image_id"].map(md5_results)
gagal = df_all["md5"].isna().sum()
if gagal > 0:
    print(f"{gagal} file gagal di-hash, diskip.")
    df_all = df_all.dropna(subset=["md5"]).reset_index(drop=True)

print(f"MD5 hash selesai untuk {len(df_all):,} gambar.")

## Langkah 5: Deteksi Exact Duplicate (MD5 Identik)
Cek apakah ada gambar yang punya MD5 hash sama persis — artinya file-nya identik.

In [ ]:
dup_mask = df_all.duplicated(subset="md5", keep=False)
df_exact_dup = df_all[dup_mask].sort_values("md5")

n_grup = df_exact_dup["md5"].nunique()
n_file = len(df_exact_dup)

print(f"Exact Duplicate (MD5 identik):")
print(f"  Grup duplikat  : {n_grup}")
print(f"  Total file     : {n_file}")

if n_grup > 0:
    cross_split = df_exact_dup.groupby("md5")["split"].nunique()
    leak_hashes = cross_split[cross_split > 1].index
    leak_df = df_exact_dup[df_exact_dup["md5"].isin(leak_hashes)]

    print(f"\n  LINTAS SPLIT (data leakage): {len(leak_hashes)} grup")
    if len(leak_hashes) > 0:
        print("  Detail gambar yang bocor antar split:")
        display(leak_df[["image_id", "split", "md5"]].head(20))
    else:
        print("  AMAN — tidak ada exact duplicate yang nyebrang antar split.")

    within_split = df_exact_dup.groupby(["md5", "split"]).size().reset_index(name="count")
    within_dup = within_split[within_split["count"] > 1]
    print(f"\n  Duplikat DALAM split yang sama: {len(within_dup)} kasus")
else:
    print("\n  BERSIH — tidak ada file yang identik sama sekali.")

## Langkah 6: Hitung Perceptual Hash (pHash)
pHash mengubah gambar jadi sidik jari visual 64-bit. Gambar yang mirip secara visual (walau beda resolusi atau kompresi) akan punya hash yang berdekatan.

In [ ]:
def hitung_phash(filepath):
    try:
        img = Image.open(filepath).convert('RGB')
        return str(imagehash.phash(img, hash_size=8))
    except Exception:
        return None

print(f"Menghitung pHash untuk {len(df_all):,} gambar...")

phash_results = {}
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(hitung_phash, row.image_path): row.image_id
               for row in df_all.itertuples()}
    done = 0
    for fut in as_completed(futures):
        img_id = futures[fut]
        phash_results[img_id] = fut.result()
        done += 1
        if done % 5000 == 0:
            print(f"  ... {done:,}/{len(df_all):,}")

df_all["phash"] = df_all["image_id"].map(phash_results)
gagal = df_all["phash"].isna().sum()
if gagal > 0:
    print(f"{gagal} gambar gagal dihitung pHash-nya, diskip.")

df_valid = df_all.dropna(subset=["phash"]).reset_index(drop=True)
print(f"pHash berhasil dihitung untuk {len(df_valid):,} gambar.")

## Langkah 7: Deteksi Near-Duplicate (pHash + Hamming Distance)
Dua gambar dianggap "hampir duplikat" kalau jarak Hamming-nya kecil (threshold ≤ 5 dari 64 bit).

Prosesnya: konversi semua pHash ke integer, lalu hitung jarak Hamming pakai XOR + popcount secara chunk supaya RAM aman.

In [ ]:
HAMMING_THRESHOLD = 5
CHUNK_SIZE = 500

hash_ints = np.array([int(h, 16) for h in df_valid["phash"]], dtype=np.uint64)
n = len(hash_ints)
print(f"Menghitung Hamming distance untuk {n:,} pasang gambar (threshold <= {HAMMING_THRESHOLD})...")

POPCOUNT_TABLE = np.zeros(256, dtype=np.int8)
for i in range(256):
    POPCOUNT_TABLE[i] = bin(i).count('1')

def hamming_matrix(chunk_ints, all_ints):
    xor = np.bitwise_xor(chunk_ints[:, None], all_ints[None, :])
    xor_bytes = xor.view(np.uint8).reshape(xor.shape[0], xor.shape[1], 8)
    return POPCOUNT_TABLE[xor_bytes].sum(axis=-1)

near_dup_pairs = []
for start in range(0, n, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n)
    dist = hamming_matrix(hash_ints[start:end], hash_ints)
    for local_i in range(end - start):
        global_i = start + local_i
        matches = np.where(dist[local_i] <= HAMMING_THRESHOLD)[0]
        for j in matches:
            if j > global_i:
                near_dup_pairs.append((global_i, j, int(dist[local_i, j])))
    if start % 5000 == 0 and start > 0:
        print(f"  ... proses {start:,}/{n:,}")

print(f"\nTotal pasangan near-duplicate (Hamming <= {HAMMING_THRESHOLD}): {len(near_dup_pairs):,}")

## Langkah 8: Analisis Near-Duplicate Lintas Split (Data Leakage)
Ini bagian kritis — kalau ada gambar mirip yang satu di train, satu lagi di val/test, berarti model bisa "nyontek" dan hasilnya gak valid.

In [ ]:
if len(near_dup_pairs) == 0:
    print("BERSIH — tidak ada near-duplicate sama sekali.")
else:
    near_df = pd.DataFrame(near_dup_pairs, columns=["idx_a", "idx_b", "hamming"])

    near_df["id_a"]    = near_df["idx_a"].map(df_valid["image_id"])
    near_df["id_b"]    = near_df["idx_b"].map(df_valid["image_id"])
    near_df["split_a"] = near_df["idx_a"].map(df_valid["split"])
    near_df["split_b"] = near_df["idx_b"].map(df_valid["split"])

    near_df["cross_split"] = near_df["split_a"] != near_df["split_b"]

    n_cross = near_df["cross_split"].sum()
    n_same  = (~near_df["cross_split"]).sum()

    print(f"Near-duplicate dalam split yang sama : {n_same:,} pasang")
    print(f"Near-duplicate LINTAS SPLIT (LEAKAGE): {n_cross:,} pasang")

    if n_cross > 0:
        print("\nContoh pasangan yang bocor antar split:")
        display(near_df[near_df["cross_split"]][["id_a", "split_a", "id_b", "split_b", "hamming"]].head(20))
    else:
        print("\nAMAN — semua near-duplicate berada di split yang sama, tidak ada kebocoran.")

## Langkah 9: Ringkasan Akhir

In [ ]:
print("=" * 60)
print("RINGKASAN AUDIT DUPLIKAT & DATA LEAKAGE")
print("=" * 60)
print(f"Dataset           : {DATASET_TYPE}")
print(f"Total gambar      : {len(df_all):,}")
print(f"Hamming threshold : {HAMMING_THRESHOLD}")
print()
print("--- Exact Duplicate (MD5 Hash) ---")
print(f"  Grup duplikat   : {n_grup}")
print(f"  Leakage lintas split : {'ADA' if n_grup > 0 and len(leak_hashes) > 0 else 'TIDAK ADA'}")
print()
if len(near_dup_pairs) > 0:
    print("--- Near-Duplicate (pHash + Hamming) ---")
    print(f"  Total pasangan  : {len(near_dup_pairs):,}")
    print(f"  Lintas split    : {n_cross:,}")
    print(f"  Leakage         : {'ADA' if n_cross > 0 else 'TIDAK ADA'}")
else:
    print("--- Near-Duplicate (pHash + Hamming) ---")
    print("  Tidak ada near-duplicate ditemukan.")
print()
print("=" * 60)
has_leak = (n_grup > 0 and len(leak_hashes) > 0) or (len(near_dup_pairs) > 0 and n_cross > 0)
if has_leak:
    print("STATUS: ADA POTENSI DATA LEAKAGE — perlu ditindaklanjuti.")
else:
    print("STATUS: BERSIH — dataset bebas dari duplikat dan leakage.")
print("=" * 60)